# Notebook 03 — Classical Knowledge Distillation

ResNet-20 is trained using the Classical KD objective (Hinton et al., 2015) — a combination of cross-entropy on hard labels and KL divergence against the teacher's soft output distributions. Temperature $T$ is ablated across $\{2, 4, 8, 16\}$ to identify the optimal setting.

**Teacher (ResNet-110):** 72.95% &nbsp;|&nbsp; **Student baseline (no KD):** 67.97% &nbsp;|&nbsp; **Gap:** 4.98%

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE  = '/content/drive/MyDrive/kd_project'
CKPT_DIR    = f'{DRIVE_BASE}/checkpoints'
RESULTS_DIR = f'{DRIVE_BASE}/results'

os.makedirs(CKPT_DIR,    exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

TEACHER_CKPT = f'{CKPT_DIR}/teacher_resnet110.pth'
assert os.path.exists(TEACHER_CKPT), \
    f'Teacher checkpoint not found at {TEACHER_CKPT}. Run Notebook 01 first.'

print('Drive mounted.')
print(f'Teacher checkpoint found: {TEACHER_CKPT}')

## 2. Environment Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU.')

## 3. Data Loading

Identical transforms to Notebooks 01 and 02 for consistency.

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5071, 0.4867, 0.4408),
                         std=(0.2675, 0.2565, 0.2761))
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5071, 0.4867, 0.4408),
                         std=(0.2675, 0.2565, 0.2761))
])

train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True,
                                               download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR100(root='./data', train=False,
                                               download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128,
                                            shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=128,
                                            shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples : {len(train_dataset):,}')
print(f'Test samples  : {len(test_dataset):,}')

## 4. Model Architecture

Same ResNet implementation as Notebooks 01 and 02.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1    = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_channels)
        self.relu     = nn.ReLU(inplace=True)
        self.conv2    = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    def forward(self, x):
        out  = self.relu(self.bn1(self.conv1(x)))
        out  = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

class ResNet_CIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=100):
        super(ResNet_CIFAR, self).__init__()
        self.in_channels = 16
        self.conv1   = nn.Conv2d(3, 16, 3, 1, 1, bias=False)
        self.bn1     = nn.BatchNorm2d(16)
        self.relu    = nn.ReLU(inplace=True)
        self.layer1  = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2  = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3  = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(64 * block.expansion, num_classes)
        self._init_weights()
    def _make_layer(self, block, out_ch, num_blocks, stride):
        layers = []
        for s in [stride] + [1]*(num_blocks-1):
            layers.append(block(self.in_channels, out_ch, s))
            self.in_channels = out_ch
        return nn.Sequential(*layers)
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01); nn.init.constant_(m.bias, 0)
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out); out = self.layer2(out); out = self.layer3(out)
        out = self.avgpool(out); out = torch.flatten(out, 1)
        return self.fc(out)

def ResNet110(num_classes=100): return ResNet_CIFAR(BasicBlock, [18,18,18], num_classes)
def ResNet20(num_classes=100):  return ResNet_CIFAR(BasicBlock, [3, 3, 3],  num_classes)

print('Architecture defined.')

## 5. Load Teacher

The teacher checkpoint from Notebook 01 is loaded and frozen. Its weights are never updated — it only provides soft probability distributions during student training.

In [ ]:
teacher = ResNet110(num_classes=100).to(device)
teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'Teacher loaded and frozen — {teacher_params/1e6:.2f}M parameters.')

## 6. KD Loss Function

The loss combines hard-label cross-entropy with KL divergence between the student and teacher softened logits. Temperature $T$ controls how soft the distributions are. The $T^2$ multiplier compensates for smaller gradient magnitudes at high temperature.

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{CE}(z_s, y) + (1-\alpha) \cdot T^2 \cdot \mathcal{L}_{KL}\!\left(\tfrac{z_s}{T},\, \tfrac{z_t}{T}\right)$$

In [ ]:
def kd_loss(student_logits, teacher_logits, labels, T, alpha):
    # Hard loss: standard cross-entropy against ground truth labels
    hard_loss = F.cross_entropy(student_logits, labels)

    # Soft loss: KL divergence between softened student and teacher distributions
    # T^2 multiplier compensates for reduced gradient magnitude at high temperature
    soft_student = F.log_softmax(student_logits / T, dim=1)
    soft_teacher = F.softmax(teacher_logits    / T, dim=1)
    soft_loss    = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * (T ** 2)

    return alpha * hard_loss + (1.0 - alpha) * soft_loss


print('KD loss function defined.')

## 7. Training and Evaluation Functions

In [ ]:
def train_kd_epoch(student, teacher, loader, optimizer, T, alpha, device):
    student.train()
    teacher.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.no_grad():
            teacher_logits = teacher(images)
        student_logits = student(images)
        loss = kd_loss(student_logits, teacher_logits, labels, T, alpha)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += (student_logits.argmax(dim=1) == labels).sum().item()
        total      += images.size(0)
    return total_loss / total, 100.0 * correct / total


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
            total   += images.size(0)
    return 100.0 * correct / total


print('Functions defined.')

## 8. Training Configuration

In [ ]:
NUM_EPOCHS    = 100
LEARNING_RATE = 0.1
MOMENTUM      = 0.9
WEIGHT_DECAY  = 5e-4
ALPHA         = 0.9   # Weight for hard loss (following Hinton et al.)
TEMPERATURES  = [2, 4, 8, 16]

print(f'Epochs       : {NUM_EPOCHS}')
print(f'Alpha        : {ALPHA}')
print(f'Temperatures : {TEMPERATURES}')
print(f'Total runs   : {len(TEMPERATURES)} x {NUM_EPOCHS} epochs')

## 9. Temperature Ablation

A fresh ResNet-20 is trained for each temperature value with all other settings fixed. Each run saves its best checkpoint to Drive.

In [ ]:
all_results = {}

for T in TEMPERATURES:
    print(f'\n{"="*65}')
    print(f'Classical KD  |  T={T}  |  Alpha={ALPHA}')
    print(f'{"="*65}')

    student   = ResNet20(num_classes=100).to(device)
    optimizer = optim.SGD(student.parameters(), lr=LEARNING_RATE,
                          momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[50, 75], gamma=0.1)
    CKPT_PATH = f'{CKPT_DIR}/kd_student_T{T}.pth'

    train_accs, test_accs = [], []
    best_acc   = 0.0
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = train_kd_epoch(
            student, teacher, train_loader, optimizer, T, ALPHA, device)
        test_acc = evaluate(student, test_loader, device)
        scheduler.step()

        train_accs.append(train_acc)
        test_accs.append(test_acc)

        if test_acc > best_acc:
            best_acc = test_acc
            torch.save(student.state_dict(), CKPT_PATH)

        elapsed    = (time.time() - start_time) / 60
        current_lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}]  '
              f'Train Acc: {train_acc:.2f}%  '
              f'Test Acc: {test_acc:.2f}%  '
              f'Best: {best_acc:.2f}%  '
              f'LR: {current_lr:.4f}  '
              f'Elapsed: {elapsed:.1f}m')

    all_results[T] = {'train_accs': train_accs, 'test_accs': test_accs, 'best_acc': best_acc}
    elapsed = (time.time() - start_time) / 60
    print(f'T={T} done — Best: {best_acc:.2f}% — Time: {elapsed:.1f}m')

print('\nAll runs complete.')

## 10. Results Summary

In [ ]:
TEACHER_ACC  = 72.95
BASELINE_ACC = 67.97

print('=' * 60)
print('  Classical KD — Temperature Ablation Results')
print('=' * 60)
print(f'  {"Model":<30} {"Best Acc":>10} {"Gap":>10}')
print(f'  {"-"*50}')
print(f'  {"Teacher (ResNet-110)":<30} {TEACHER_ACC:>9.2f}% {"—":>10}')
print(f'  {"Baseline (no KD)":<30} {BASELINE_ACC:>9.2f}% {"-4.98%":>10}')
print(f'  {"-"*50}')
for T in TEMPERATURES:
    acc = all_results[T]['best_acc']
    gap = acc - TEACHER_ACC
    print(f'  {f"KD T={T}":<30} {acc:>9.2f}% {gap:>+9.2f}%')
print('=' * 60)

best_T   = max(all_results, key=lambda t: all_results[t]['best_acc'])
best_acc = all_results[best_T]['best_acc']
print(f'\nBest temperature : T={best_T} — {best_acc:.2f}%')
print(f'Gap closed       : +{best_acc - BASELINE_ACC:.2f}% of the {TEACHER_ACC - BASELINE_ACC:.2f}% gap')

## 11. Training Curves — All Temperatures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes      = axes.flatten()
epochs_range = range(1, NUM_EPOCHS + 1)
colors       = ['steelblue', 'tomato', 'seagreen', 'darkorange']

for i, T in enumerate(TEMPERATURES):
    ax = axes[i]
    ax.plot(epochs_range, all_results[T]['train_accs'],
            label='Train', color=colors[i], linewidth=1.5)
    ax.plot(epochs_range, all_results[T]['test_accs'],
            label='Test',  color=colors[i], linewidth=1.5, linestyle='--')
    ax.axvline(x=50, color='gray', linestyle=':', alpha=0.7)
    ax.axvline(x=75, color='gray', linestyle=':', alpha=0.7)
    ax.axhline(y=BASELINE_ACC, color='red',   linestyle='--', alpha=0.5,
               linewidth=1, label=f'Baseline ({BASELINE_ACC:.2f}%)')
    ax.axhline(y=TEACHER_ACC,  color='black', linestyle='--', alpha=0.5,
               linewidth=1, label=f'Teacher ({TEACHER_ACC:.2f}%)')
    ax.set_title(f'T={T}  |  Best: {all_results[T]["best_acc"]:.2f}%', fontsize=11)
    ax.set_xlabel('Epoch');  ax.set_ylabel('Accuracy (%)')
    ax.legend(fontsize=8);   ax.grid(True, alpha=0.3)
    ax.set_ylim([40, 100])

plt.suptitle('Classical KD — Temperature Ablation (T = 2, 4, 8, 16)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/kd_temperature_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to Drive.')

## 12. Temperature Comparison Bar Chart

In [ ]:
temps = list(all_results.keys())
accs  = [all_results[T]['best_acc'] for T in temps]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(T) for T in temps], accs,
              color='steelblue', edgecolor='black', linewidth=0.8)

best_idx = accs.index(max(accs))
bars[best_idx].set_color('tomato')

ax.axhline(y=BASELINE_ACC, color='red',   linestyle='--', linewidth=1.5,
           label=f'Baseline (no KD): {BASELINE_ACC:.2f}%')
ax.axhline(y=TEACHER_ACC,  color='black', linestyle='--', linewidth=1.5,
           label=f'Teacher: {TEACHER_ACC:.2f}%')

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Temperature T', fontsize=12)
ax.set_ylabel('Best Test Accuracy (%)', fontsize=12)
ax.set_title('Classical KD — Best Test Accuracy by Temperature', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim([65, 75])
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/kd_temperature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to Drive.')

## 13. Final Evaluation — Best Temperature

In [ ]:
best_T    = max(all_results, key=lambda t: all_results[t]['best_acc'])
best_ckpt = f'{CKPT_DIR}/kd_student_T{best_T}.pth'

best_student = ResNet20(num_classes=100).to(device)
best_student.load_state_dict(torch.load(best_ckpt, map_location=device))
final_acc = evaluate(best_student, test_loader, device)

print('=' * 50)
print('  Classical KD — Final Results')
print('=' * 50)
print(f'  Teacher (ResNet-110)     : {TEACHER_ACC:.2f}%')
print(f'  Student baseline (no KD) : {BASELINE_ACC:.2f}%')
print(f'  Classical KD (T={best_T})    : {final_acc:.2f}%')
print(f'  Gap closed               : +{final_acc - BASELINE_ACC:.2f}%')
print(f'  Remaining gap to teacher : {TEACHER_ACC - final_acc:.2f}%')
print('=' * 50)
print(f'  Best checkpoint: {best_ckpt}')